In [7]:
pip install yfinance

In [8]:

import requests
import pandas as pd
import time
import csv
from pathlib import Path
from datetime import datetime, timezone
from io import StringIO

Path("logs").mkdir(exist_ok=True)

def record_pipeline_run(status, message):
    log_path = Path("logs/pipeline_events.csv")
    new_file = not log_path.exists()

    with open(log_path, "a", newline="") as file:
        writer = csv.writer(file)

        if new_file:
            writer.writerow([
                "timestamp_utc",
                "status",
                "message"
            ])

        writer.writerow([
            datetime.now(timezone.utc).isoformat(),
            status,
            message
        ])

def download_fred_with_retry(url, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()

            data = pd.read_csv(StringIO(response.text))

            if data.empty or not {"observation_date", "DFF"}.issubset(data.columns):
                raise ValueError("Empty data or unexpected FRED columns")

            record_pipeline_run(
                "SUCCESS",
                f"FRED download succeeded on attempt {attempt}"
            )

            return data

        except (requests.RequestException, ValueError) as error:
            record_pipeline_run(
                "RETRY_FAILURE",
                f"Attempt {attempt}: {error}"
            )

            if attempt < attempts:
                time.sleep(2)

    record_pipeline_run(
        "FAILED",
        "FRED download failed after all attempts"
    )

    raise RuntimeError("FRED download failed after all attempts")

print("PASS: Retry and logging functions created!")


PASS: Retry and logging functions created!


In [9]:

import yfinance as yf
import time

def download_sp500_with_retry(attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            data = yf.Ticker("^GSPC").history(period="6mo")

            if data.empty:
                raise ValueError("S&P 500 returned empty data")

            record_pipeline_run(
                "SUCCESS",
                f"S&P 500 downloaded on attempt {attempt}"
            )
            return data

        except Exception as error:
            record_pipeline_run(
                "RETRY_FAILURE",
                f"S&P 500 attempt {attempt}: {error}"
            )

            if attempt < attempts:
                time.sleep(2)

    record_pipeline_run(
        "FAILED",
        "S&P 500 download failed after all attempts"
    )

    raise RuntimeError("S&P 500 download failed")

print("PASS: S&P 500 retry function created!")


PASS: S&P 500 retry function created!


In [10]:

import yfinance as yf

sp_data = download_sp500_with_retry()
df = sp_data.copy()

df.to_csv("sp500_data.csv")

print("PASS: S&P 500 data downloaded and saved!")


PASS: S&P 500 data downloaded and saved!


In [11]:
print(df.head())
print("Total rows:", len(df))
print("Columns:", df.columns.tolist())

                                  Open         High          Low        Close  \
Date                                                                            
2026-04-30 00:00:00-04:00  7161.750000  7219.830078  7126.149902  7209.009766   
2026-05-01 00:00:00-04:00  7234.540039  7272.520020  7229.319824  7230.120117   
2026-05-04 00:00:00-04:00  7228.379883  7244.540039  7174.120117  7200.750000   
2026-05-05 00:00:00-04:00  7233.620117  7273.259766  7233.620117  7259.220215   
2026-05-06 00:00:00-04:00  7294.140137  7369.220215  7294.140137  7365.120117   

                               Volume  Dividends  Stock Splits  
Date                                                            
2026-04-30 00:00:00-04:00  5723790000        0.0           0.0  
2026-05-01 00:00:00-04:00  4847390000        0.0           0.0  
2026-05-04 00:00:00-04:00  5215480000        0.0           0.0  
2026-05-05 00:00:00-04:00  5279760000        0.0           0.0  
2026-05-06 00:00:00-04:00  6579640000     

In [12]:
from pathlib import Path

#Create a folder for original data
Path("data/raw").mkdir(parents=True, exist_ok=True)

#Save the downloaded data
df.to_csv("data/raw/sp500_raw.csv")

print("Raw S&P 500 data saved successfully!")

Raw S&P 500 data saved successfully!


In [13]:
#Check the number of missing values in each column
print("Missing values in each column:")
print(df.isnull().sum())

#Check how many rows and columns we have
print("\nDataset size:")
print(df.shape)

Missing values in each column:
Open            0
High            0
Low             0
Close           0
Volume          0
Dividends       0
Stock Splits    0
dtype: int64

Dataset size:
(128, 7)


In [14]:
# Create a copy of the original dataset
test_data = df.copy()

print("Original rows:", len(df))
print("Copied rows:", len(test_data))

Original rows: 128
Copied rows: 128


In [15]:
# Deliberately remove one closing price
test_data.iloc[0, test_data.columns.get_loc("Close")] = None

#Display the first five rows
print(test_data.head())

                                  Open         High          Low        Close  \
Date                                                                            
2026-04-30 00:00:00-04:00  7161.750000  7219.830078  7126.149902          NaN   
2026-05-01 00:00:00-04:00  7234.540039  7272.520020  7229.319824  7230.120117   
2026-05-04 00:00:00-04:00  7228.379883  7244.540039  7174.120117  7200.750000   
2026-05-05 00:00:00-04:00  7233.620117  7273.259766  7233.620117  7259.220215   
2026-05-06 00:00:00-04:00  7294.140137  7369.220215  7294.140137  7365.120117   

                               Volume  Dividends  Stock Splits  
Date                                                            
2026-04-30 00:00:00-04:00  5723790000        0.0           0.0  
2026-05-01 00:00:00-04:00  4847390000        0.0           0.0  
2026-05-04 00:00:00-04:00  5215480000        0.0           0.0  
2026-05-05 00:00:00-04:00  5279760000        0.0           0.0  
2026-05-06 00:00:00-04:00  6579640000     

In [16]:
#Find the rows with missing or invalid closing prices
invalid_rows = test_data[
    test_data["Close"].isna() |
    (test_data["Close"] <= 0)
].copy()

print("Number of invalid records:", len(invalid_rows))
print(invalid_rows)

Number of invalid records: 1
                              Open         High          Low  Close  \
Date                                                                  
2026-04-30 00:00:00-04:00  7161.75  7219.830078  7126.149902    NaN   

                               Volume  Dividends  Stock Splits  
Date                                                            
2026-04-30 00:00:00-04:00  5723790000        0.0           0.0  


In [17]:
from pathlib import Path
from datetime import datetime, timezone

# Create a folder for error logs
Path("logs").mkdir(exist_ok=True)

# Add details explaining the problem
invalid_rows["error_reason"] = "Missing or invalid closing price"

invalid_rows["logged_at_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

#Save the invalid records
invalid_rows.to_csv(
    "logs/data_quality_issues.csv"
)

print("Error log saved successfully!")
print("Errors recorded:", len(invalid_rows))

Error log saved successfully!
Errors recorded: 1


In [18]:
valid_rows = test_data[
    test_data["Close"].notna() &
    (test_data["Close"] > 0)
].copy()

#Create a folder for validated data
Path("data/curated").mkdir(parents=True, exist_ok=True)

# Saved validated records
valid_rows.to_csv(
    "data/curated/sp500_validated.csv"
)

print("Valid records saved:", len(valid_rows))
print("Invalid records logged:", len(invalid_rows))

Valid records saved: 127
Invalid records logged: 1


In [19]:

import shutil
from google.colab import files

# Package your data and logs into one ZIP file
shutil.make_archive(
    "BUSA3022_Data_Pipeline_Files",
    "zip",
    ".",
    "data"
)

# Download the ZIP to your computer
files.download("BUSA3022_Data_Pipeline_Files.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:

# Start with the original downloaded data
real_data = df.copy()

# Check for missing or invalid closing prices
invalid_real = real_data[
    real_data["Close"].isna() |
    (real_data["Close"] <= 0)
].copy()

# Keep only valid records
valid_real = real_data[
    real_data["Close"].notna() &
    (real_data["Close"] > 0)
].copy()

# Save the real validated dataset
valid_real.to_csv(
    "data/curated/sp500_validated.csv"
)

# Display the results
print("Original records:", len(real_data))
print("Valid records:", len(valid_real))
print("Invalid records:", len(invalid_real))

assert len(real_data) == len(valid_real) + len(invalid_real)
print("PASS: All records accounted for!")


Original records: 128
Valid records: 128
Invalid records: 0
PASS: All records accounted for!


In [21]:

import pandas as pd

# Check whether any dates are repeated
duplicate_dates = valid_real.index.duplicated().sum()

# Check whether any dates are missing
missing_dates = valid_real.index.isna().sum()

# Check whether dates are in chronological order
dates_sorted = valid_real.index.is_monotonic_increasing

print("Duplicate dates:", duplicate_dates)
print("Missing dates:", missing_dates)
print("Dates in chronological order:", dates_sorted)

# Confirm the dataset passes all checks
assert duplicate_dates == 0
assert missing_dates == 0
assert dates_sorted

print("PASS: Date validation successful!")


Duplicate dates: 0
Missing dates: 0
Dates in chronological order: True
PASS: Date validation successful!


In [22]:

import pandas as pd
from pathlib import Path

# FRED's daily Effective Federal Funds Rate
fred_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DFF"

# Download the dataset
fred_data = pd.read_csv(fred_url)

# Display the first five rows
print(fred_data.head())
print("Total rows:", len(fred_data))


  observation_date   DFF
0       1954-07-01  1.13
1       1954-07-02  1.25
2       1954-07-03  1.25
3       1954-07-04  1.25
4       1954-07-05  0.88
Total rows: 26419


In [23]:

import pandas as pd
from pathlib import Path

# FRED's daily Effective Federal Funds Rate
fred_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DFF"

# Download the dataset
fred_data = pd.read_csv(fred_url)

# Display the first five rows
print(fred_data.head())
print("Total rows:", len(fred_data))


  observation_date   DFF
0       1954-07-01  1.13
1       1954-07-02  1.25
2       1954-07-03  1.25
3       1954-07-04  1.25
4       1954-07-05  0.88
Total rows: 26419


In [24]:

import pandas as pd
from pathlib import Path

# Create a copy so the original remains unchanged
fred_clean = fred_data.copy()

# Convert dates into proper date values
fred_clean["observation_date"] = pd.to_datetime(
    fred_clean["observation_date"],
    errors="coerce"
)

# Convert interest rates into numbers
fred_clean["DFF"] = pd.to_numeric(
    fred_clean["DFF"],
    errors="coerce"
)

# Find missing or invalid records
invalid_fred = fred_clean[
    fred_clean["observation_date"].isna() |
    fred_clean["DFF"].isna()
].copy()

# Keep valid records
valid_fred = fred_clean.drop(
    invalid_fred.index
).copy()

print("Original records:", len(fred_data))
print("Valid records:", len(valid_fred))
print("Invalid records:", len(invalid_fred))

assert len(fred_data) == len(valid_fred) + len(invalid_fred)

print("PASS: All FRED records accounted for!")


Original records: 26419
Valid records: 26419
Invalid records: 0
PASS: All FRED records accounted for!


In [25]:

# Create curated storage if needed
Path("data/curated").mkdir(parents=True, exist_ok=True)

# Save validated FRED records
valid_fred.to_csv(
    "data/curated/fred_validated.csv",
    index=False
)

print("Validated FRED data saved successfully!")


Validated FRED data saved successfully!


In [26]:

# Check FRED for duplicate dates
duplicate_fred = valid_fred["observation_date"].duplicated().sum()

# Check for missing dates
missing_fred_dates = valid_fred["observation_date"].isna().sum()

# Check chronological order
fred_sorted = valid_fred["observation_date"].is_monotonic_increasing

print("Duplicate FRED dates:", duplicate_fred)
print("Missing FRED dates:", missing_fred_dates)
print("Dates in chronological order:", fred_sorted)

assert duplicate_fred == 0
assert missing_fred_dates == 0
assert fred_sorted

print("PASS: FRED date validation successful!")


Duplicate FRED dates: 0
Missing FRED dates: 0
Dates in chronological order: True
PASS: FRED date validation successful!


In [27]:

from pathlib import Path
from datetime import datetime, timezone

Path("logs").mkdir(exist_ok=True)

# Record any invalid FRED observations
fred_errors = invalid_fred.copy()

fred_errors["source"] = "FRED"
fred_errors["error_reason"] = "Missing or invalid date/rate"
fred_errors["logged_at_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

# Save a separate FRED quality log
fred_errors.to_csv(
    "logs/fred_quality_issues.csv",
    index=False
)

print("FRED quality log created!")
print("Invalid FRED records:", len(fred_errors))


FRED quality log created!
Invalid FRED records: 0


In [28]:

# Make a separate copy for testing
fred_test = fred_data.copy()

# Deliberately introduce a missing interest rate
fred_test.loc[0, "DFF"] = None

print(fred_test.head())


  observation_date   DFF
0       1954-07-01   NaN
1       1954-07-02  1.25
2       1954-07-03  1.25
3       1954-07-04  1.25
4       1954-07-05  0.88


In [29]:

import pandas as pd
from datetime import datetime, timezone

# Convert test values to numbers
fred_test["DFF"] = pd.to_numeric(
    fred_test["DFF"], errors="coerce"
)

# Identify missing or invalid rates
fred_test_errors = fred_test[
    fred_test["DFF"].isna()
].copy()

# Record why the rows failed
fred_test_errors["source"] = "FRED"
fred_test_errors["error_reason"] = "Missing interest rate"
fred_test_errors["test_type"] = "Simulated failure"
fred_test_errors["logged_at_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

# Save a separate test log
fred_test_errors.to_csv(
    "logs/fred_simulated_errors.csv",
    index=False
)

print("Simulated errors detected:", len(fred_test_errors))

assert len(fred_test_errors) >= 1
print("PASS: FRED error detection successful!")


Simulated errors detected: 1
PASS: FRED error detection successful!


In [30]:

import requests
import time
import logging
from pathlib import Path

# Create folder for system logs
Path("logs").mkdir(exist_ok=True)

# Configure logging
logging.basicConfig(
    filename="logs/api_failures.log",
    level=logging.ERROR,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

# Deliberately use an invalid URL for testing
test_url = "https://invalid-source.example/data.csv"

# Attempt to retrieve data up to three times
for attempt in range(1, 4):
    try:
        response = requests.get(test_url, timeout=5)
        response.raise_for_status()

        print("API connection successful!")
        break

    except requests.RequestException as error:
        print(f"Attempt {attempt} failed: {error}")

        logging.error(
            "Simulated API failure on attempt %s: %s",
            attempt, error
        )

        if attempt < 3:
            time.sleep(1)
else:
    print("All attempts failed. Error recorded in log.")


Attempt 1 failed: HTTPSConnectionPool(host='invalid-source.example', port=443): Max retries exceeded with url: /data.csv (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7ff56bf60ad0>: Failed to resolve 'invalid-source.example' ([Errno -2] Name or service not known)"))
Attempt 2 failed: HTTPSConnectionPool(host='invalid-source.example', port=443): Max retries exceeded with url: /data.csv (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7ff56bfb2710>: Failed to resolve 'invalid-source.example' ([Errno -2] Name or service not known)"))
Attempt 3 failed: HTTPSConnectionPool(host='invalid-source.example', port=443): Max retries exceeded with url: /data.csv (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7ff56bfb3110>: Failed to resolve 'invalid-source.example' ([Errno -2] Name or service not known)"))
All attempts failed. Error recorded in log.


In [31]:

from pathlib import Path

log_file = Path("logs/api_failures.log")

# Check the log file exists
assert log_file.exists(), "ERROR: Log file not found!"

# Read the log
log_contents = log_file.read_text()

# Check that all three failures were recorded
for attempt in range(1, 4):
    assert f"Simulated API failure on attempt {attempt}" in log_contents

print("PASS: API failure log verified!")
print("All three failed attempts were recorded.")


PASS: API failure log verified!
All three failed attempts were recorded.


In [32]:

import pandas as pd
import yfinance as yf
from pathlib import Path
from datetime import datetime, timezone

# Create storage folders
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/curated").mkdir(parents=True, exist_ok=True)
Path("logs").mkdir(exist_ok=True)

print("Starting BUSA3022 data pipeline...")

# 1. Download S&P 500 data
sp_data = download_sp500_with_retry()

if sp_data.empty:
    raise ValueError("S&P 500 download returned no data")


# 2. Validate S&P 500 data
sp_invalid = (
    sp_data["Close"].isna() |
    (sp_data["Close"] <= 0) |
    sp_data.index.isna() |
    sp_data.index.duplicated()
)

sp_valid = sp_data.loc[~sp_invalid].copy()
sp_errors = sp_data.loc[sp_invalid].copy()


# 3. Download FRED data
fred_url = (
    "https://fred.stlouisfed.org/graph/"
    "fredgraph.csv?id=DFF"
)

fred_raw = download_fred_with_retry(fred_url)

# 4. Validate FRED data
fred_clean = fred_raw.copy()

fred_clean["observation_date"] = pd.to_datetime(
    fred_clean["observation_date"], errors="coerce"
)

fred_clean["DFF"] = pd.to_numeric(
    fred_clean["DFF"], errors="coerce"
)

fred_invalid = (
    fred_clean["observation_date"].isna() |
    fred_clean["DFF"].isna() |
    fred_clean["observation_date"].duplicated()
)

fred_valid = fred_clean.loc[~fred_invalid].copy()
fred_errors = fred_clean.loc[fred_invalid].copy()



# 5. Save quality logs
sp_errors.to_csv("logs/sp500_quality_issues.csv")
fred_errors.to_csv(
    "logs/fred_quality_issues.csv",
    index=False
)

# 6. Display pipeline summary
print("\nPIPELINE SUMMARY")
print("S&P 500 valid records:", len(sp_valid))
print("S&P 500 invalid records:", len(sp_errors))
print("FRED valid records:", len(fred_valid))
print("FRED invalid records:", len(fred_errors))

assert len(sp_data) == len(sp_valid) + len(sp_errors)
assert len(fred_clean) == len(fred_valid) + len(fred_errors)


# 7. Safely publish validated datasets
import os
import shutil
import tempfile
from pathlib import Path

files_to_save = {
    "data/raw/sp500_raw.csv": (sp_data, True),
    "data/raw/fred_raw.csv": (fred_raw, False),
    "data/curated/sp500_validated.csv": (sp_valid, True),
    "data/curated/fred_validated.csv": (fred_valid, False)
}

# Prepare all files before changing final outputs
with tempfile.TemporaryDirectory() as staging_dir:
    staged = {}

    for destination, (dataset, keep_index) in files_to_save.items():
        staged_path = Path(staging_dir) / Path(destination).name
        dataset.to_csv(staged_path, index=keep_index)
        staged[destination] = staged_path

    # Save copies of existing files for rollback
    with tempfile.TemporaryDirectory() as rollback_dir:
        previous = {}

        for destination in files_to_save:
            target = Path(destination)
            backup = Path(rollback_dir) / target.name

            if target.exists():
                shutil.copy2(target, backup)
                previous[destination] = backup
            else:
                previous[destination] = None

        try:
            for destination, staged_path in staged.items():
                target = Path(destination)
                target.parent.mkdir(parents=True, exist_ok=True)
                os.replace(staged_path, target)

        except Exception:
            # Restore the previous files if publishing fails
            for destination, backup in previous.items():
                target = Path(destination)

                if backup is not None:
                    shutil.copy2(backup, target)
                elif target.exists():
                    target.unlink()

            raise

print("PASS: All four datasets saved successfully!")


Starting BUSA3022 data pipeline...

PIPELINE SUMMARY
S&P 500 valid records: 128
S&P 500 invalid records: 0
FRED valid records: 26419
FRED invalid records: 0
PASS: All four datasets saved successfully!


In [33]:

import csv
from pathlib import Path
from datetime import datetime, timezone

# Create a log recording this successful pipeline run
log_path = Path("logs/pipeline_runs.csv")

# Check whether the log already exists
new_file = not log_path.exists()

with open(log_path, "a", newline="") as file:
    writer = csv.writer(file)

    if new_file:
        writer.writerow([
            "timestamp_utc",
            "status",
            "sp500_valid",
            "sp500_invalid",
            "fred_valid",
            "fred_invalid"
        ])

    writer.writerow([
        datetime.now(timezone.utc).isoformat(),
        "SUCCESS",
        len(sp_valid),
        len(sp_errors),
        len(fred_valid),
        len(fred_errors)
    ])

print("PASS: Pipeline execution logged!")


PASS: Pipeline execution logged!


In [34]:

fred_url = (
    "https://fred.stlouisfed.org/graph/"
    "fredgraph.csv?id=DFF"
)

try:
    fred_retry_data = download_fred_with_retry(fred_url)

    print("FRED download successful!")
    print("Records retrieved:", len(fred_retry_data))

except RuntimeError as error:
    print("Download failed:", error)


FRED download successful!
Records retrieved: 26419


In [35]:

sp_retry_data = download_sp500_with_retry()

print("S&P 500 download successful!")
print("Records retrieved:", len(sp_retry_data))

S&P 500 download successful!
Records retrieved: 128


In [36]:

from pathlib import Path
import shutil

# Create a backup folder
Path("data/backups").mkdir(parents=True, exist_ok=True)

# Files we want to protect
files_to_backup = [
    "data/raw/sp500_raw.csv",
    "data/raw/fred_raw.csv",
    "data/curated/sp500_validated.csv",
    "data/curated/fred_validated.csv"
]

# Copy each file into the backup folder
for file in files_to_backup:
    source = Path(file)

    if source.exists():
        destination = Path("data/backups") / source.name

        shutil.copy2(source, destination)

        print("BACKUP CREATED:", source.name)
    else:
        print("WARNING: File not found:", file)

print("Backup process finished!")


BACKUP CREATED: sp500_raw.csv
BACKUP CREATED: fred_raw.csv
BACKUP CREATED: sp500_validated.csv
BACKUP CREATED: fred_validated.csv
Backup process finished!


In [37]:

from pathlib import Path
import hashlib

files = [
    "data/raw/sp500_raw.csv",
    "data/raw/fred_raw.csv",
    "data/curated/sp500_validated.csv",
    "data/curated/fred_validated.csv"
]

# Record the contents of all four saved files
before = {}

for file in files:
    path = Path(file)
    assert path.exists(), f"Missing file: {file}"
    before[file] = hashlib.sha256(path.read_bytes()).hexdigest()

# Simulate a failed download without modifying files
try:
    raise RuntimeError("Simulated API download failure")
except RuntimeError as error:
    print("Expected failure:", error)

# Verify every saved file remains unchanged
for file in files:
    after = hashlib.sha256(Path(file).read_bytes()).hexdigest()
    assert before[file] == after, f"File changed: {file}"

print("PASS: All four saved datasets remain unchanged!")


Expected failure: Simulated API download failure
PASS: All four saved datasets remain unchanged!


In [38]:

# Test the existing FRED retry function
# without changing any saved datasets

from pathlib import Path
import hashlib

files = [
    "data/raw/sp500_raw.csv",
    "data/raw/fred_raw.csv",
    "data/curated/sp500_validated.csv",
    "data/curated/fred_validated.csv"
]

before = {
    file: hashlib.sha256(Path(file).read_bytes()).hexdigest()
    for file in files
}

# Deliberately use an invalid URL
try:
    download_fred_with_retry(
        "https://invalid-source.example/fred.csv"
    )
    print("FAIL: Expected download failure was not raised")
except RuntimeError:
    print("PASS: FRED download failure detected")

# Confirm saved files were not changed
for file in files:
    current = hashlib.sha256(Path(file).read_bytes()).hexdigest()
    assert current == before[file], f"File changed: {file}"

print("PASS: All four saved datasets remain unchanged!")


PASS: FRED download failure detected
PASS: All four saved datasets remain unchanged!


In [39]:

from pathlib import Path
import hashlib
import json

files = [
    "data/raw/sp500_raw.csv",
    "data/raw/fred_raw.csv",
    "data/curated/sp500_validated.csv",
    "data/curated/fred_validated.csv"
]

checkpoint = {}

for file in files:
    path = Path(file)
    assert path.exists(), f"Missing file: {file}"
    checkpoint[file] = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

Path("data/backups").mkdir(
    parents=True, exist_ok=True
)

with open("data/backups/checkpoint.json", "w") as f:
    json.dump(checkpoint, f, indent=2)

print("PASS: Dataset checkpoint created!")
print("Files protected by checkpoint:", len(checkpoint))


PASS: Dataset checkpoint created!
Files protected by checkpoint: 4


In [40]:

from pathlib import Path
import tempfile
import os
import hashlib

with tempfile.TemporaryDirectory() as folder:
    target = Path(folder) / "test.csv"

    # Simulate an existing successful dataset
    target.write_text("date,price\n2026-10-30,100\n")

    original_hash = hashlib.sha256(
        target.read_bytes()
    ).hexdigest()

    # Simulate a failed pipeline before publishing
    staged = Path(folder) / "test.tmp"

    try:
        staged.write_text("date,price\n2026-10-31,200\n")

        # Simulate a validation failure
        raise ValueError("Simulated validation failure")

        os.replace(staged, target)

    except ValueError as error:
        print("Expected failure:", error)
        staged.unlink(missing_ok=True)

    final_hash = hashlib.sha256(
        target.read_bytes()
    ).hexdigest()

    assert original_hash == final_hash

    print("PASS: Previous dataset preserved!")
    print("PASS: Failed staged file discarded!")


Expected failure: Simulated validation failure
PASS: Previous dataset preserved!
PASS: Failed staged file discarded!


In [41]:

from pathlib import Path
import hashlib
import json

# Load the original checkpoint
with open("data/backups/checkpoint.json", "r") as f:
    checkpoint = json.load(f)

# Verify all four datasets
for filename, original_hash in checkpoint.items():
    path = Path(filename)

    assert path.exists(), f"Missing file: {filename}"

    current_hash = hashlib.sha256(
        path.read_bytes()
    ).hexdigest()

    assert current_hash == original_hash, (
        f"Dataset changed: {filename}"
    )

    print("PASS:", filename)

print("\nPASS: All four datasets verified!")


PASS: data/raw/sp500_raw.csv
PASS: data/raw/fred_raw.csv
PASS: data/curated/sp500_validated.csv
PASS: data/curated/fred_validated.csv

PASS: All four datasets verified!


In [42]:

from pathlib import Path
import zipfile

handoff_files = [
    "data/raw/sp500_raw.csv",
    "data/raw/fred_raw.csv",
    "data/curated/sp500_validated.csv",
    "data/curated/fred_validated.csv"
]

zip_name = "BUSA3022_Member2_Data_Handoff.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for file in handoff_files:
        assert Path(file).exists(), f"Missing: {file}"
        z.write(file)
        print("Added:", file)

print("\nPASS: Member 2 handoff ZIP created!")


Added: data/raw/sp500_raw.csv
Added: data/raw/fred_raw.csv
Added: data/curated/sp500_validated.csv
Added: data/curated/fred_validated.csv

PASS: Member 2 handoff ZIP created!


BUSA3022 — Data Pipeline Architecture
Data Sources

Yahoo Finance — S&P 500 (^GSPC)

FRED — Federal Funds Rate (DFF)


Python Ingestion

yfinance / FRED CSV endpoint

Retry handling and API failure logging


Raw Storage

data/raw/

Original downloaded datasets


Validation and Transformation

Missing values · Invalid values · Duplicate dates

Quality issues recorded separately


Curated Storage

data/curated/

Validated S&P 500 and FRED datasets


Member 2 — Data Curation

Date alignment · Metrics · Further transformations

Supporting controls: logs/ for quality and API failures; data/backups/ for backup and checkpoint files.